In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [2]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [3]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [4]:
q = ground_truth[0]
q

{'question': 'I just found this course late — can I still join, or is it too late to start?',
 'document': '74eb249bbf'}

In [5]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [6]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
04919992b3 == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
9f689c185f == 74eb249bbf: False


In [7]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [8]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [9]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I just found this course late — can I still join, or is it too late to start?


[1, 0, 0, 0, 0]

In [10]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [11]:
ground_truth_sample = ground_truth
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/515 [00:00<?, ?it/s]

In [12]:
relevance_total_text[:5]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [13]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [14]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [15]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)

  0%|          | 0/515 [00:00<?, ?it/s]

In [16]:
relevance_total[:10]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [17]:
cnt = 0

for line in relevance_total:
    if 1 in line:
        cnt = cnt + 1

cnt

433

In [18]:
cnt / len(relevance_total)

0.8407766990291262

In [19]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [20]:
hit_rate(relevance_total)

0.8407766990291262

In [21]:
total_score = 0.0

for line in relevance_total:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

375.43333333333317

In [22]:
total_score / len(relevance_total)

0.728996763754045

In [23]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [24]:
mrr(relevance_total)

0.728996763754045

In [25]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [26]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/515 [00:00<?, ?it/s]

{'hit_rate': 0.8407766990291262, 'mrr': 0.728996763754045}

In [28]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [29]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/515 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8815533980582524, 'mrr': 0.7818122977346276}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.887378640776699, 'mrr': 0.7771844660194173}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8407766990291262, 'mrr': 0.728996763754045}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8174757281553398, 'mrr': 0.701650485436893}


  0%|          | 0/515 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.8, 'mrr': 0.6730097087378638}


In [30]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [31]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/515 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/515 [00:00<?, ?it/s]

In [32]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
6,1.0,4.0,0.1,0.959223,0.862783
34,5.0,10.0,0.2,0.961165,0.861650
33,5.0,10.0,0.1,0.961165,0.861489
3,1.0,2.0,0.1,0.963107,0.861262
35,5.0,10.0,0.5,0.963107,0.861262
19,2.0,4.0,0.2,0.963107,0.861262
18,2.0,4.0,0.1,0.961165,0.860777
7,1.0,4.0,0.2,0.957282,0.860615
4,1.0,2.0,0.2,0.961165,0.857605
22,2.0,10.0,0.2,0.957282,0.856214


In [33]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )